## Collect Events
Used for mapping catorgory i.e. sports, econ, politics

In [ ]:
import requests
import pandas as pd
import time

url_type = 'events'
base_url = f"https://api.elections.kalshi.com/trade-api/v2/{url_type}"

all_events_data = []
cursor = None
max_pages = 10  # Increased to ensure you get all categories (Economics, Sports, etc.)

print(f"Fetching all {url_type} data from Kalshi...")

for page in range(max_pages):
    params = {
        "status": "open",
        "limit": 200,
        "cursor": cursor
    }

    response = requests.get(base_url, params=params)
    
    if response.status_code != 200:
        print(f"Error {response.status_code}: {response.text}")
        break
        
    data = response.json()
    batch = data.get(url_type, [])
    all_events_data.extend(batch)
    
    # Update cursor for the next page
    cursor = data.get('cursor')
    print(f"Page {page+1} fetched. Total events so far: {len(all_events_data)}")
    
    if not cursor:
        print("Reached the end of the available events.")
        break
        
    time.sleep(0.5)

events_df = pd.DataFrame(all_events_data)

# 3. Save to CSV
# events_df.to_csv('data/kalshi_events.csv', index=False)

# 4. Quick Stats for your awareness
print("\n--- Data Summary ---")
print(f"Shape: {events_df.shape}")

if 'category' in events_df.columns:
    print("\nBreakdown by Category:")
    print(events_df['category'].value_counts())

# Display the first few rows of the raw data
print(events_df.head())

Fetching all events data from Kalshi...
Page 1 fetched. Total events so far: 200
Page 2 fetched. Total events so far: 400
Page 3 fetched. Total events so far: 600
Page 4 fetched. Total events so far: 800
Page 5 fetched. Total events so far: 1000
Page 6 fetched. Total events so far: 1200
Page 7 fetched. Total events so far: 1400
Page 8 fetched. Total events so far: 1600
Page 9 fetched. Total events so far: 1800
Page 10 fetched. Total events so far: 2000

--- Data Summary ---
Shape: (2000, 12)

Breakdown by Category:
category
Elections                 953
Sports                    261
Politics                  227
Entertainment             212
Economics                 170
Companies                  53
Science and Technology     35
Financials                 21
Crypto                     18
Social                     17
World                      14
Climate and Weather        13
Health                      5
Transportation              1
Name: count, dtype: int64
   available_on_brokers 

## Collect Markets
This is the pricing data

In [ ]:
url_type = 'markets'
base_url = f"https://api.elections.kalshi.com/trade-api/v2/{url_type}"

all_market_data = []
cursor = None
max_pages = 5  # Increase this to get even more data (e.g., 10 or 20)

print("Fetching data from Kalshi...")
for page in range(max_pages):
    params = {
        'mve_filter': 'exclude',
        'status': 'open',
        'limit': 1000,  # Maximize the limit to reduce API calls
        'cursor': cursor
    }

    response = requests.get(base_url, params=params)
    
    if response.status_code != 200:
        print(f"Error {response.status_code}: {response.text}")
        break
        
    data = response.json()
    batch = data.get(url_type, [])
    all_market_data.extend(batch)
    

    cursor = data.get('cursor')
    print(f"Page {page+1} complete. Total records: {len(all_market_data)}")
    
    # If no more pages exist, stop
    if not cursor:
        break
    
    time.sleep(0.5)

market_df = pd.DataFrame(all_market_data)

cols_to_keep = [
    'ticker',                # Unique ID for the specific contract
    'event_ticker',          # Unique ID for the parent event
    'title',                 # High-level event name
    'subtitle',              # Specific outcome description
    'yes_sub_title',         # The "Yes" side label
    'yes_ask_dollars',       # Buy price for Yes
    'yes_bid_dollars',       # Sell price for Yes
    'no_sub_title',          # The "No" side label
    'no_ask_dollars',        # Buy price for No
    'no_bid_dollars',        # Sell price for No
    'market_type',           # Usually 'binary'
    'early_close_condition', # Useful for matching
    'rules_primary',         # Settlement logic
    'rules_secondary',       # Edge case logic
    'close_time',            # When betting stops
    'expiration_time',       # When the market settles
    'can_close_early'        # Risk factor for Arb
]

# Ensure we only keep columns that actually exist in the data
final_cols = [c for c in cols_to_keep if c in market_df.columns]
market_df = market_df[final_cols]

# 4. Save and Verify
# market_df.to_csv('data/kalshi_market.csv', index=False)

print("\n--- Verification ---")
print(f"Final Shape: {market_df.shape}")

Fetching data from Kalshi...
Page 1 complete. Total records: 1000
Page 2 complete. Total records: 2000
Page 3 complete. Total records: 3000
Page 4 complete. Total records: 4000
Page 5 complete. Total records: 5000

--- Verification ---
Final Shape: (5000, 17)


## Backfill Data
In collecting for both market, and events, it does not always match exact tickers, so we need to extract those specific missing tickers

In [74]:
# 1. Identify which event_tickers we are missing info for
all_market_tickers = market_df['event_ticker'].unique()
known_event_tickers = events_df['event_ticker'].unique()
missing_tickers = [t for t in all_market_tickers if t not in known_event_tickers]

print(f"Total Unique Event Tickers in Markets: {len(all_market_tickers)}")
print(f"Missing from current Events DataFrame: {len(missing_tickers)}")

# 2. Backfill the missing events (Targeted API calls)
backfilled_events = []
if len(missing_tickers) > 0:
    print("Backfilling missing event data...")
    for ticker in missing_tickers:
        # Get specific event details
        url = f"https://api.elections.kalshi.com/trade-api/v2/events/{ticker}"
        res = requests.get(url)
        
        if res.status_code == 200:
            # The API returns {'event': {...}}, so we extract the dict
            event_data = res.json().get('event', {})
            backfilled_events.append(event_data)
        
        # Rate limiting: Kalshi is generous, but let's be safe
        time.sleep(0.05) 

# 3. Combine old events with newly backfilled ones
if backfilled_events:
    new_events_df = pd.DataFrame(backfilled_events)
    full_events_df = pd.concat([events_df, new_events_df], ignore_index=True).drop_duplicates('event_ticker')
else:
    full_events_df = events_df

# 4. Final Merge
combined_df = pd.merge(
    market_df, 
    full_events_df, 
    on='event_ticker', 
    how='inner',
    suffixes=('', '_event')
)

print(f"\n--- SUCCESS ---")
print(f"New Combined Shape: {combined_df.shape}")
print(f"Categories Captured: {combined_df['category'].unique() if 'category' in combined_df.columns else 'None'}")
combined_df.to_csv('data/kalshi_master_data.csv', index=False)

Total Unique Event Tickers in Markets: 553
Missing from current Events DataFrame: 543
Backfilling missing event data...

--- SUCCESS ---
New Combined Shape: (5000, 28)
Categories Captured: <StringArray>
[             'Crypto',              'Sports',       'Entertainment',
 'Climate and Weather',            'Mentions',           'Economics',
          'Financials',            'Politics',           'Elections']
Length: 9, dtype: str


In [75]:
combined_df

,ticker,event_ticker,title,subtitle,yes_sub_title,yes_ask_dollars,yes_bid_dollars,no_sub_title,no_ask_dollars,no_bid_dollars,...,category,collateral_return_type,last_updated_ts,mutually_exclusive,series_ticker,strike_period,sub_title,title_event,product_metadata,strike_date
0,KXHYPE-26MAR2417-T55.9999,KXHYPE-26MAR2417,"HYPE price on Mar 24, 2026?",,$56 or above,0.0000,0.0000,$56 or above,1.0000,1.0000,...,Crypto,MECNET,0001-01-01T00:00:00Z,True,KXHYPE,,"On Mar 24, 2026 at 5pm EDT","HYPE price on Mar 24, 2026 at 5pm EDT?",NaN,2026-03-24T21:00:00Z
1,KXHYPE-26MAR2417-T18.00,KXHYPE-26MAR2417,"HYPE price on Mar 24, 2026?",,$17.9999 or below,0.0000,0.0000,$17.9999 or below,1.0000,1.0000,...,Crypto,MECNET,0001-01-01T00:00:00Z,True,KXHYPE,,"On Mar 24, 2026 at 5pm EDT","HYPE price on Mar 24, 2026 at 5pm EDT?",NaN,2026-03-24T21:00:00Z
2,KXHYPE-26MAR2417-B55,KXHYPE-26MAR2417,"HYPE price on Mar 24, 2026?",,$55 to 55.9999,0.0000,0.0000,$55 to 55.9999,1.0000,1.0000,...,Crypto,MECNET,0001-01-01T00:00:00Z,True,KXHYPE,,"On Mar 24, 2026 at 5pm EDT","HYPE price on Mar 24, 2026 at 5pm EDT?",NaN,2026-03-24T21:00:00Z
3,KXHYPE-26MAR2417-B54,KXHYPE-26MAR2417,"HYPE price on Mar 24, 2026?",,$54 to 54.9999,0.0000,0.0000,$54 to 54.9999,1.0000,1.0000,...,Crypto,MECNET,0001-01-01T00:00:00Z,True,KXHYPE,,"On Mar 24, 2026 at 5pm EDT","HYPE price on Mar 24, 2026 at 5pm EDT?",NaN,2026-03-24T21:00:00Z
4,KXHYPE-26MAR2417-B53,KXHYPE-26MAR2417,"HYPE price on Mar 24, 2026?",,$53 to 53.9999,0.0000,0.0000,$53 to 53.9999,1.0000,1.0000,...,Crypto,MECNET,0001-01-01T00:00:00Z,True,KXHYPE,,"On Mar 24, 2026 at 5pm EDT","HYPE price on Mar 24, 2026 at 5pm EDT?",NaN,2026-03-24T21:00:00Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,KXXRP-26MAR2417-B1.1899500,KXXRP-26MAR2417,"Ripple price at Mar 24, 2026 at 5pm EDT?",,$1.18 to 1.1999,0.0300,0.0000,$1.18 to 1.1999,1.0000,0.9700,...,Crypto,MECNET,0001-01-01T00:00:00Z,True,KXXRP,,"On Mar 24, 2026 at 5pm EDT","Ripple price range at Mar 24, 2026 at 5pm EDT?","{'competition': 'XRP', 'competition_scope': 'H...",2026-03-24T21:00:00Z
4996,KXXRP-26MAR2417-B1.1699500,KXXRP-26MAR2417,"Ripple price at Mar 24, 2026 at 5pm EDT?",,$1.16 to 1.1799,0.0300,0.0000,$1.16 to 1.1799,1.0000,0.9700,...,Crypto,MECNET,0001-01-01T00:00:00Z,True,KXXRP,,"On Mar 24, 2026 at 5pm EDT","Ripple price range at Mar 24, 2026 at 5pm EDT?","{'competition': 'XRP', 'competition_scope': 'H...",2026-03-24T21:00:00Z
4997,KXXRP-26MAR2417-B1.1499500,KXXRP-26MAR2417,"Ripple price at Mar 24, 2026 at 5pm EDT?",,$1.14 to 1.1599,0.0300,0.0000,$1.14 to 1.1599,1.0000,0.9700,...,Crypto,MECNET,0001-01-01T00:00:00Z,True,KXXRP,,"On Mar 24, 2026 at 5pm EDT","Ripple price range at Mar 24, 2026 at 5pm EDT?","{'competition': 'XRP', 'competition_scope': 'H...",2026-03-24T21:00:00Z
4998,KXXRP-26MAR2417-B1.1299500,KXXRP-26MAR2417,"Ripple price at Mar 24, 2026 at 5pm EDT?",,$1.12 to 1.1399,0.0100,0.0000,$1.12 to 1.1399,1.0000,0.9900,...,Crypto,MECNET,0001-01-01T00:00:00Z,True,KXXRP,,"On Mar 24, 2026 at 5pm EDT","Ripple price range at Mar 24, 2026 at 5pm EDT?","{'competition': 'XRP', 'competition_scope': 'H...",2026-03-24T21:00:00Z
